Before we begin, let us execute the below cell to display information about the NVIDIA® CUDA® driver and the GPUs running on the server by running the `nvidia-smi` command. To do this, execute the cell block below by clicking on it with your mouse, and pressing Ctrl+Enter, or pressing the play button in the toolbar above. You should see some output returned below the grey cell.

In [ ]:
nvidia-smi

# Learning objectives
The **goal** of this lab is to:

- Apply parallelisation techniques to loops using standard keywords in C++ and Fortran across both CPU and GPU architectures.
- Examine methods for safely managing shared variables by using atomic operations to prevent race conditions.
- Utilise the concept and benefits of unified memory for efficient data sharing between CPU and GPU.
- Employ modern C++ features, including lambda expressions, to improve the readability and maintainability of code.
- Enhance parallel performance by implementing techniques such as loop collapsing.
- Select appropriate compiler flags to compile parallel Fortran code for both CPU and GPU targets.

We do not intend to cover:
- Detailed optimization techniques and mapping of standard constructs to CUDA C

**NOTE**: To be able to see the Nsight Systems profiler output, please download the latest version of Nsight Systems from [here](https://developer.nvidia.com/nsight-systems).

# Standard Template Library (STL)

If you are not familiar with STL (Standard Template Library), this section will give you a brief introduction that would be required to understand the usage of STL library for our code.

The C++ STL (Standard Template Library) is a powerful set of C++ template classes to provide general-purpose classes and functions with templates that implement many popular and commonly used algorithms and data structures like vectors, lists, queues, and stacks.

At the core of the C++ Standard Template Library are following three well-structured components 

- Containers: Containers are used to manage collections of objects of a certain kind. There are several different types of containers like vector, map, deque, list etc.

- Algorithms: Algorithms act on containers. They provide the means by which you will perform initialization, sorting, searching, and transforming of the contents of containers.

- Iterators: Iterators are used to step through the elements of collections of objects. These collections may be containers or subsets of containers.

First we will introduce you to the container and how to use an iterator to step through elements of a vector. The `vector` container (a C++ Standard Template) is similar to an array with an exception that it automatically handles its own storage requirements in case it grows.

To iterate through a container, we will use the `std::for_each` algorithm. A sample usage is shown in the code below:

```c++
#include <vector>
#include <algorithm>
#include <iostream>
 
//Using functor
struct Sum
{
    void operator()(int n) { sum += n; }
    int sum{0};
};
 
int main()
{
    std::vector<int> nums{3, 4, 2, 8, 15, 267};
 
    auto print = [](const int& n) { std::cout << " " << n; };
 
    std::cout << "before:";
    std::for_each(nums.cbegin(), nums.cend(), print);
    std::cout << '\n';
 
    std::for_each(nums.begin(), nums.end(), [](int &n){ n++; });
 
    // calls Sum::operator() for each number
    Sum s = std::for_each(nums.begin(), nums.end(), Sum());
 
    std::cout << "after: ";
    std::for_each(nums.cbegin(), nums.cend(), print);
    std::cout << '\n';
    std::cout << "sum: " << s.sum << '\n';
}
```

To learn more about STL you can read and execute sample codes [here](https://www.tutorialspoint.com/cplusplus/cpp_stl_tutorial.htm).

# Parallel STL
Starting with C++17, parallelism has become an integral part of the standard itself. Parallel STL is an implementation of the C++ standard library algorithms with support for execution policies, commonly called C++17.

C++17 Parallel Standard Library (stdpar) introduces parallel and vector concurrency for standard algorithms. It is important to note that stdpar is a library and not a language extension.


## `std::par` Execution Policies


Execution Policies define the kind of parallelism that will be applied to parallel algorithms. Most standard algorithms included in STL support execution policies. Defined below are the execution policies:

- `std::execution::seq` = sequential
    - This execution policy type is used as a unique type to disambiguate parallel algorithm overloading and requires that a parallel algorithm’s execution may not be parallelized.
- `std::execution::par` = parallel
    - This execution policy type is used as a unique type to disambiguate parallel algorithm overloading and indicate that a parallel algorithm’s execution may be parallelized
- `std::execution::par_unseq` = parallel + vectorized
    - This execution policy type used as a unique type to disambiguate parallel algorithm overloading and indicate that a parallel algorithm’s execution may be parallelized and vectorized

Implementation of execution policies is provided by different compilers from specific vendors. For GPU parallel execution policy we will be making use of the NVIDIA compiler. 


## Historical Perspective

Changes to how the call to _stl_ algorithms changed the new version of C++ standard to incorporate execution policies:

**C++98:** 
```c++
std::sort(c.begin(), c.end()); 
```
**C++17:** 
```c++
std::sort(std::execution::par, c.begin(), c.end());
```

We will be using the NVIDIA HPC C++ compiler, NVC++. It supports C++17, C++ Standard Parallelism (stdpar) for NVIDIA GPUs, OpenACC for multicore CPUs and NVIDIA GPUs, and OpenMP for multicore CPUs. No language extensions or non-standard libraries are required to enable GPU acceleration. All data movement between host memory and GPU device memory is performed implicitly and automatically under the control of CUDA Unified Memory, which means that heap memory is automatically shared between a CPU(Host) and GPU(Device). Stack memory and global memory are not shared. Below given example shows the right allocation and usage of the stdpar.

```c++
std::vector<int> v = ...;
std::sort(std::execution::par, v.begin(), v.end()); // OK, vector allocates on heap

std::array<int, 1024> a = ...;
std::sort(std::execution::par, a.begin(), a.end()); // Fails, array stored on the stack
```

For our code we will be making use of `std::for_each` algorithm with support for `std::execution::par` execution policy.  However, as we will require to iterate through multiple arrays to read in the data and write to a separate array, we will not utilise simple containers in our `std::for_each` algorithm but instead iterate over the input arrays using a special `counting_iterator`.

**Counting Iterator**: This iterator represents a pointer into a range of sequentially changing values. This iterator is useful for creating a range filled with a sequence without explicitly storing it in memory. Using `counting_iterator` saves memory capacity and bandwidth.  In the exercise we will use the `std::iota` to create the `counting_iterator`.

```c++
auto res = std::ranges::views::iota(range_start, range_end);
```

## Atomic operations

In the code, you will also require one more template container, which will help you get the right results. The SLT atomic type, `std::atomic<T>`, ensures that a particular variable is accessed and/or updated atomically to prevent indeterminate results and race conditions. In other words, it prevents one thread from stepping on the toes of other threads due to accessing a variable simultaneously, resulting in different results run-to-run. For example, if we want to accumulate numbers up to N, we could write the following:

```c++
std::atomic<int> cnt;

for (int i = 0; i < N; ++i){
    cnt.fetch_add(1, std::memory_order_relaxed);
    // This is equivalent to ++cnt; but explicitly telling the compiler 
    // to avoid unnecessary synchronization.
}
```


# Standard Exercise

Lets start modifying the original code and add the necessary changes to parallelise the code. Without changing the orginal code, you will get error running the below cells.

**Click on the <b>[**source code**](../source_code/rdf.cpp)</b> link, and start modifying the RDF code.**

To help you modify the code, some sections are marked with `TODO: ` comments consisting of simple instructions.  Where additional modifications from the [original source code](../../_common/source_code/rdf.cpp) were required they are marked with `Note: ` comments for you to review. Remember to **SAVE** your code after changes, before running the below cells.

**NOTE**: Using the STL approach in C++ can often require restructuring some of the code, which will depend on if and how the existing code uses the STL.

## Multicore

Now, let's compile the code. We will be using NVIDIA HPC SDK for this exercise. The flags used for enabling standard parallelism for target offloading are as follows:

- `-stdpar` : This flag enables standard parallelism for the target architecture
- `-stdpar=multicore` will allow us to compile our code for a multicore
- `-stdpar` will allow us to compile our code for a NVIDIA GPU (Default is NVIDIA)

After running the cells, you can inspect part of the compiler feedback and see what it's telling us (your compiler feedback will be similar to the below).

### Compile and run the code

In [ ]:
#Compile the code for multicore
cd ../source_code && printf "Compiling for multicore ...\n" && make clean && make rdf_c && 
printf "\nRunning the executable and validating the output\n" &&./rdf_c && cat Pair_entropy.dat

The output should be the following:
    
```
s2 value is -2.43191
s2bond value is -3.87015
    
```

and an example compiler feedback would look similar as below:
    
```
main:
     70, stdpar: Generating Multicore code
         70, std::fill with std::execution::par policy parallelized on CPU
pair_gpu(float const*, float const*, float const*, std::atomic<int>*, int, int, double, double, double, double, double):
    167, stdpar: Generating Multicore code
        167, std::for_each with std::execution::par policy parallelized on CPU
```

In [ ]:
#profile and see output of nvptx
cd ../source_code && nsys profile -t nvtx --stats=true --force-overwrite true -o rdf_stdpar_multicore ./rdf_c

Let's checkout the profiler's report. Download and save the report file by holding down the Shift key and right-clicking the [report link](../source_code/rdf_stdpar_multicore.nsys-rep) then choosing Save Link As. Once done, open it via the GUI. From the _Timeline View_, checkout the NVTX markers displays as part of threads. **Why are we using NVTX?** Please see the Moodle section on [Using NVIDIA Tools Extension (NVTX)](https://129.234.196.13/moodle/mod/page/view.php?id=29).

From the _Timeline View_, right click on the nvtx row and click the "show in events view". Now you can see the nvtx statistic at the bottom of the window which shows the duration of each range. 

**Example screenshot (multicore)**

<img src="../../_common/images/stdpar_multicore.png">


## NVIDIA GPU

Without changing the code now let us try to recompile the code for NVIDIA GPU and rerun. GPU acceleration of standard parallel algorithms is enabled with the `-⁠stdpar` command-line option when using NVIDIA HPC C++ compiler. If `-⁠stdpar `is specified, almost all algorithms that use a parallel execution policy are compiled for offloading to run in parallel on an NVIDIA GPU.

**Understand and analyze** the [solution](../source_code/SOLUTION/rdf.cpp) and compare with your version. Once done, compile your code for GPU by running below cells.

### Compile and run the code

In [ ]:
#compile for Tesla GPU
cd ../source_code && echo "Compiling for GPU ... " && nvc++ -std=c++20 -stdpar=gpu -Minfo=stdpar -o rdf_c rdf.cpp && 
printf "\nRunning the executable and validating the output\n" && ./rdf_c && cat Pair_entropy.dat


The output should be the following:
   
```
s2 value is -2.43191
s2bond value is -3.87015
    
```
    
and an example compiler feedback would look similar as below:
    
```
main:
     70, stdpar: Generating NVIDIA GPU code
         70, std::fill with std::execution::par policy parallelized on GPU
pair_gpu(float const*, float const*, float const*, std::atomic<int>*, int, int, double, double, double, double, double):
    167, stdpar: Generating NVIDIA GPU code
        167, std::for_each with std::execution::par policy parallelized on GPU
```


In [ ]:
#profile and see output of nvptx
cd ../source_code && nsys profile -t nvtx,cuda --stats=true --force-overwrite true -o rdf_stdpar_gpu ./rdf_c

Let's checkout the profiler's report. Download and save the report file by holding down the Shift key and right-clicking the [report link](../source_code/rdf_stdpar_gpu.nsys-rep) then choosing Save Link As. Once done, open it via the GUI. 

From the "_Timeline View_" on the top pane, double click on the "CUDA" from the function table on the left and expand it. Hover your mouse over the CUDA row (underlined with blue color in the below screenshot) and expand it till you see both kernels and memory row.  Zoom in on the timeline and you can see a pattern similar to the screenshot below. The blue boxes (annotated with a red box) are the compute kernels. The small red and green boxes (annotated with a green box) represent data movements.  Similarly, expanding the "Threads" reveals a "CUDA API" (annoted with a purple box) which shows the different CUDA API used. Notice the CudaMallocManaged API indicating the use of Unified (managed) memory.

**Example screenshot (GPU)**

<img src="../../_common/images/stdpar_gpu.png">

If you inspect the output of the profiler closer, you can see the *Unified Memory* usage. Moreover, if you compare the NVTX marker `Pair_Calculation` (from the NVTX row) in both multicore and GPU version, you can see how much improvement you achieved. 

Feel free to checkout the [solution](../source_code/SOLUTION/rdf.cpp) again to help you understand better.

# Standard Language Analysis

## Usage Scenarios
stdpar is part of the standard language, all C++ compilers are expected to support it moving forward. This is C++ standard-compliant code, so you can maintain a single codebase. It provides a good start for accelerating code on accelerators like GPU and multicores.

## Limitations/Constraints
1. This isn’t a catch-all solution or necessarily an alternative to other programming models that provide more control over things like thread management. *std:par* provides the highest portability and can be seen as the first step to porting on GPU. The general abstraction limits the optimization functionalities. For example, the implementations are currently dependent on Unified memory. Moreover, one does not have control over thread management and that will limit performance improvement.
2. C++ constructs can only be used in the code using C++17 features and may not work for legacy codes.

## Which Compilers Support stdpar on GPUs and Multicore?
1. NVIDIA GPU: As of Jan 2021, the HPC SDK compiler from NVIDIA supports std::par and DO-CONCURRENT on NVIDIA GPU.
2. x86 Multicore: stdpar: GCC has an implementation on a multicore CPU which is based on Intel TBB in the backend

# Saving the exercise

If you would like to download this exercise for later viewing, it is recommended you go to your browser's file menu (not the Jupyter notebook file menu) and save the complete web page.  This will ensure the images are copied down as well. You can also execute the following cell block to create a zip file of the files you have been working on, and download it with the link below.

In [ ]:
cd ..
rm -f _files.zip
zip -r _files.zip *

**After** executing the above zip command, you should be able to download and save the zip file by holding down Shift and right-clicking [Here](../_files.zip) then choosing save Link As.

# Links and Resources
[Blog post on Developing Accelerated Code with Standard Language Parallelism](https://developer.nvidia.com/blog/developing-accelerated-code-with-standard-language-parallelism/)

[Blog post on Accelerating Standard C++ with GPUs Using stdpar](https://developer.nvidia.com/blog/accelerating-standard-c-with-gpus-using-stdpar/)

[NVIDIA Nsight System](https://docs.nvidia.com/nsight-systems/)

# Licensing 

Copyright © 2022 OpenACC-Standard.org.  This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.